# 34 — Queries, Keys, Values, and Scaled Attention

**Master syllabus coverage:** V2 5.4–5.5

## Why this matters

Attention is the Transformer's communication mechanism. Deriving every axis and invariant now prevents opaque errors when heads, masks, caching, and optimized kernels are added.

## Learning objectives

- Project residual states into queries, keys, and values.
- Derive score, weight, and output shapes for one attention head.
- Explain scaling by square root of head dimension statistically.
- Apply a causal mask before softmax and match PyTorch SDPA.

Work through the notebook in order. Predict shapes and results before running each code cell. An assertion failure is part of the lesson: read it before changing code.


## 1. Attention is content-addressed weighted retrieval

Each token produces a query (what am I looking for?), key (what do I contain?), and
value (what information do I send?). For one head:

$$Q=XW_Q,\quad K=XW_K,\quad V=XW_V,$$
$$A=\operatorname{softmax}(QK^\top/\sqrt D),\quad Y=AV.$$

Shapes: `X=[B,T,C]`, projections `[C,D]`, `Q,K,V=[B,T,D]`, scores/weights
`[B,Tq,Tk]`, output `[B,T,D]`.


In [ ]:
import math
import torch

torch.manual_seed(42)
B, T, C, D = 2, 4, 6, 3
X = torch.randn(B, T, C)
Wq, Wk, Wv = (torch.randn(C, D) for _ in range(3))
Q, K, V = X @ Wq, X @ Wk, X @ Wv
scores = Q @ K.transpose(-2, -1) / math.sqrt(D)
weights = torch.softmax(scores, dim=-1)
output = weights @ V
print("Q/K/V:", Q.shape, "scores:", scores.shape, "output:", output.shape)
torch.testing.assert_close(weights.sum(dim=-1), torch.ones(B, T))


## 2. Why divide by $\sqrt D$?

If query/key components are independent with unit variance, their dot product has
variance $D$. Growing $D$ makes logits large, softmax overly sharp, and gradients small.
Dividing by $\sqrt D$ keeps score variance near one at initialization.


In [ ]:
for head_dim in (8, 32, 128, 512):
    q = torch.randn(20_000, head_dim)
    k = torch.randn(20_000, head_dim)
    raw = (q * k).sum(dim=-1)
    scaled = raw / math.sqrt(head_dim)
    print(f"D={head_dim:3}: raw std={raw.std():6.2f}, scaled std={scaled.std():5.2f}")


## 3. Attention output is a convex combination per query

Softmax weights are nonnegative and sum to one. Each output row lies in the convex hull
of the value rows for that head. Learned output projection and residual addition later
mix heads and recover a full-width update.


In [ ]:
simple_values = torch.tensor([[[0., 0.], [2., 0.], [0., 4.]]])  # [1,3,2]
simple_weights = torch.tensor([[[0.25, 0.25, 0.50]]])            # [1,1,3]
retrieved = simple_weights @ simple_values                       # [1,1,2]
print("retrieved:", retrieved)
torch.testing.assert_close(retrieved, torch.tensor([[[0.5, 2.0]]]))


## 4. Mask before softmax

Disallowed scores receive `-inf` (or the minimum safe representable value) before
softmax, yielding zero probability. Multiplying probabilities by a mask afterward breaks
row normalization unless renormalized and may allow information to affect earlier steps.
A fully masked row has no valid categorical distribution and can produce NaN.


In [ ]:
causal_allowed = torch.tril(torch.ones(T, T, dtype=torch.bool))
masked_scores = scores.masked_fill(~causal_allowed, float("-inf"))
causal_weights = torch.softmax(masked_scores, dim=-1)
causal_output = causal_weights @ V
assert not causal_weights.triu(diagonal=1).any()
torch.testing.assert_close(causal_weights.sum(dim=-1), torch.ones(B, T))
print("first query weights:", causal_weights[0, 0])


## 5. Match PyTorch's scaled dot-product attention

Framework kernels may fuse masking, softmax, dropout, and value aggregation. First prove
semantic equivalence on a small deterministic example; performance work comes later.


In [ ]:
reference = torch.nn.functional.scaled_dot_product_attention(
    Q[:, None], K[:, None], V[:, None], is_causal=True, dropout_p=0.0
).squeeze(1)
torch.testing.assert_close(causal_output, reference, atol=1e-6, rtol=1e-5)
print("manual causal attention matches PyTorch SDPA")


## Failure modes to recognize

- **Wrong transpose:** dot products mix feature or query axes incorrectly.
- **Missing scaling:** softmax saturates increasingly as head width grows.
- **Mask after softmax:** disallowed positions influenced normalization or keep nonzero mass.
- **Fully masked row:** softmax of all negative infinity becomes undefined.

These are diagnostic patterns, not trivia. When a future model fails, reduce the problem to the smallest example in this notebook and test the relevant invariant.


## Deliberate practice

1. Write all shapes for cross-attention where query and key sequence lengths differ.
2. Change one key vector and identify which score column and output rows can change.
3. Gradient-check the manual attention function in float64 on a tiny tensor.

Do not merely rerun the provided cells. Make a prediction, change one variable, and write down why the result did or did not match your prediction.

## Exit ticket

**You are ready to continue when:** you can implement causal scaled dot-product attention, annotate every contraction, and verify its probabilities and reference output.

**Next:** 35 — Causal multi-head attention.
